# WixQA RAG System

## Part 1: Baseline

### Chunking WixQA dataset

We'll prepare:
- Fixed size chunking
- Semantic Chunking

and upload to huggingface

Some modern embedder models:

- jinaai/jina-embeddings-v5-text-small
- KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2.5
- https://build.nvidia.com/nvidia/nv-embedqa-e5-v5/deploy?environment=linux.md

These are pretty heavy however, lightweight models that has library support:
- all-MiniLM-L6-v2
- BGE-Small-v1.5

Leaderboard at https://huggingface.co/spaces/mteb/leaderboard

We'll split the documents into sentences, embed them with jinaai/jina-embeddings-v5-text-small
, and evaluate similarities to eachother

In [ ]:
!pip install -q unstructured langchain_text_splitters

In [1]:
# if you are evaluating
# using pgvector
!pip install -q psycopg2 pgvector evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


#### Loading Dataset

In [2]:
from datasets import load_dataset
from typing import Dict, List, Tuple
import pandas as pd

_DATASET_REPO = "Wix/WixQA"
_KB_CONFIG = "wix_kb_corpus"
_EXPERT_CONFIG = "wixqa_expertwritten"
_SIMULATED_CONFIG = "wixqa_simulated"
_SYNTHETIC_CONFIG = "wixqa_synthetic"

def load_kb_corpus() -> List[Dict]:
    """
    id (str) - article identifier
    url (str) - public URL of the article
    contents (str) - full article text
    article_type(str) - "article" | "feature_request" | "known_issue"
    """
    ds = load_dataset(_DATASET_REPO, _KB_CONFIG, split="train")
    articles = list(ds)
    print(f"{len(articles)} articles loaded.")
    return articles


def load_qa_split(split: str) -> List[Dict]:
    """
    Load "expert", "simulated", "synthetic"
    splits from huggingface

    https://huggingface.co/datasets/Wix/WixQA
    """

    config_map = {
        "expert": _EXPERT_CONFIG,
        "simulated": _SIMULATED_CONFIG,
        "synthetic": _SYNTHETIC_CONFIG,
    }

    config = config_map[split]

    print(f"Loading WixQA-{split} from HuggingFace…")

    ds = load_dataset(_DATASET_REPO, config, split="train")
    rows = list(ds)
    print(f"{len(rows)} QA pairs loaded.")
    return rows


# lookup table {a["id"]: a for a in articles}
# convert qa rows to df pd.DataFrame(qa_rows)

In [3]:
# WixQA document corpus
articles = load_kb_corpus()

README.md: 0.00B [00:00, ?B/s]

wix_kb_corpus/wix_kb_corpus.jsonl:   0%|          | 0.00/53.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6221 [00:00<?, ? examples/s]

6221 articles loaded.


#### Preparing Chunks

- **Fixed-size tokenization**: straightforward, we
also allow overlap
- **Semantic Chunking**
  - Split by sentences
  - For each sentence, create an embedding
  - We perform similarity checks on a windowed approach
  - We set a breakpoint for when there is a lack of similarity

For Semantic chunking, we borrow ideas from langchain - smart merging, if a semantic chunk is too small (less than 300), we push it to a neighbor, and if it exceeds a 800 token limit, we force a sub-split

Issues found:

- Sentence splitters like nltk do not split well on rather unstructured and messy data like WixQA
- Splitting by sentences does not scale well. Many individual sentences are redundant on their own without possibility of surrounding context.
- The data is unclean, and we can try to preprocess the text using the unstructured library

https://docs.langchain.com/oss/python/integrations/splitters


In [4]:
import re
from typing import List
import nltk

from unstructured.cleaners.core import (
    clean,
    clean_extra_whitespace,
    group_broken_paragraphs
)

nltk.download("punkt_tab", quiet=True)
nltk.download("punkt", quiet=True)

True

In [5]:
# The WixQA content is pretty messy,
# so we need to clean the data beforehand

def clean_wixqa_text(text: str) -> str:
    if not text:
        return ""

    # if we have a Lowercase char followed by Uppercase char, we insert space
    text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)

    # if we have an instance like 'End of sentence.Next sentence',
    # we add a space, 'End of sentence. Next sentence'
    text = re.sub(r'(?<=[.!?])(?=[A-Za-z])', ' ', text)

    # fix List Headers/Artifacts
    # ex Note:Make fixed to Note: Make
    text = re.sub(r'(Note|Tips|Warning|Step \d+):', r'\1: ', text)

    # use unstructured library to clean
    text = group_broken_paragraphs(text)
    text = clean_extra_whitespace(text)
    text = clean(text, bullets=True, extra_whitespace=True, dashes=True)

    return text.strip()

def get_sentences(text: str) -> List[str]:

    # clean text
    cleaned_text = clean_wixqa_text(text)

    # get sentences from nltk
    raw_sentences = nltk.sent_tokenize(cleaned_text)

    sentences = []
    buffer = ""

    # special appreviations to avoid sentence splits on
    abbreviations = {'e.g.', 'i.e.', 'vs.', 'etc.', 'fig.', 'approx.'}

    for s in raw_sentences:
        s = s.strip()
        combined = f"{buffer} {s}".strip() if buffer else s

        # check abbreviations or unclosed parens
        ends_with_abbr = any(combined.lower().endswith(abbr) for abbr in abbreviations)
        unclosed_paren = combined.count('(') > combined.count(')')

        if ends_with_abbr or unclosed_paren:
            buffer = combined
        else:
            if len(combined) > 5:
                sentences.append(combined)
            buffer = ""

    if buffer:
        sentences.append(buffer)

    return sentences

In [6]:
from typing import Dict, List, Tuple

class Chunk:
    __slots__ = ("chunk_id", "article_id", "text", "metadata")

    def __init__(
        self,
        chunk_id: str,
        article_id: str,
        text: str,
        metadata: Dict | None = None,
    ):
        self.chunk_id = chunk_id
        self.article_id = article_id
        self.text = text.strip()
        self.metadata = metadata or {}

    def __repr__(self) -> str:
        return (
            f"Chunk(id={self.chunk_id!r}, "
            f"article={self.article_id!r}, "
            f"len={len(self.text)})"
        )

# we return fixed size chunks that have optional
# overlap with eachother
def _fixed_chunks(
    text: str, chunk_size: int = 512, overlap: int = 64
) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]


#### bdanko/wixqa_kb_corpus_chunk_500_overlap_60

In [16]:
fixed_results = []

for article in articles:
    article_id = article.get("id")
    clean_text = clean_wixqa_text(article.get("contents", ""))

    if not clean_text:
        continue

    f_chunks = _fixed_chunks(clean_text, chunk_size=500, overlap=60)
    for i, text in enumerate(f_chunks):
        fixed_results.append(Chunk(
            chunk_id=f"{article_id}_f{i}",
            article_id=article_id,
            text=text,
            metadata={"method": "fixed"}
        ))

print(f"Fixed Chunks: {len(fixed_results)}")

Fixed Chunks: 34954


In [17]:
for chunk in fixed_results[:5]:
  print(f"\nChunk:\n {chunk.text}")


Chunk:
 Wix Events: About the Event Details and Registration Form Pages Guests visiting your site view the events you offer on the Events List page. From there they can learn more on the Events Details page (if enabled) and complete the booking on the Registration Form Page. You can customize how these pages look to suit your events. Tips: Customizations you make to the Events Details page and the Registration Form page (e. g. changing colors and fonts, hiding page elements) affect these page for all ev

Chunk:
 nd fonts, hiding page elements) affect these page for all events. If your site only has events without tickets, you have the option of disabling the Event Details Page. Events Details Page Site visitors who click to register for an event are first directed to the Event Details page (unless you hid the page). There, guests can receive more complete information about the event. You can customize the page to show or hide various elements, such as a map to the event location and a

In [18]:
import pandas as pd

lengths = [len(c.text) for c in fixed_results]
dist_df = pd.Series(lengths)
print(dist_df.describe())
print("\nLength Quantiles:")
print(dist_df.quantile([0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

count    34954.000000
mean       451.604623
std        114.890832
min          1.000000
25%        499.000000
50%        500.000000
75%        500.000000
max        500.000000
dtype: float64

Length Quantiles:
0.05    141.0
0.25    499.0
0.50    500.0
0.75    500.0
0.95    500.0
0.99    500.0
dtype: float64


In [19]:
from datasets import Dataset

data_list = [
    {
        "chunk_id": c.chunk_id,
        "article_id": c.article_id,
        "text": c.text,
        "method": c.metadata.get("method")
    }
    for c in fixed_results
]

dataset = Dataset.from_list(data_list)

In [14]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

!hf auth login --token {HF_TOKEN}

A new version of huggingface_hub (1.7.1) is available! You are using version 1.6.0.
To update, run: pip install -U huggingface_hub

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `all!` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `all!`


In [20]:
dataset.push_to_hub("bdanko/wixqa_kb_corpus_chunk_500_overlap_60", private=False)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/35 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  28%|##7       | 2.19MB / 7.89MB            

CommitInfo(commit_url='https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_500_overlap_60/commit/d712282cf5f5eec5ed019cf801dce38bff19210e', commit_message='Upload dataset', commit_description='', oid='d712282cf5f5eec5ed019cf801dce38bff19210e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_500_overlap_60', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bdanko/wixqa_kb_corpus_chunk_500_overlap_60'), pr_revision=None, pr_num=None)

#### bdanko/wixqa_kb_corpus_chunk_500_overlap_30



In [22]:
fixed_results = []

for article in articles:
    article_id = article.get("id")
    clean_text = clean_wixqa_text(article.get("contents", ""))

    if not clean_text:
        continue

    f_chunks = _fixed_chunks(clean_text, chunk_size=500, overlap=30)
    for i, text in enumerate(f_chunks):
        fixed_results.append(Chunk(
            chunk_id=f"{article_id}_f{i}",
            article_id=article_id,
            text=text,
            metadata={"method": "fixed"}
        ))

print(f"Fixed Chunks: {len(fixed_results)}")

from datasets import Dataset

data_list = [
    {
        "chunk_id": c.chunk_id,
        "article_id": c.article_id,
        "text": c.text,
        "method": c.metadata.get("method")
    }
    for c in fixed_results
]

dataset = Dataset.from_list(data_list)

dataset.push_to_hub("bdanko/wixqa_kb_corpus_chunk_500_overlap_30", private=False)

Fixed Chunks: 32911


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/33 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  49%|####8     | 3.81MB / 7.79MB            

CommitInfo(commit_url='https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_500_overlap_30/commit/330894b377ef353f77c189e27ba8b788145cb0a5', commit_message='Upload dataset', commit_description='', oid='330894b377ef353f77c189e27ba8b788145cb0a5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_500_overlap_30', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bdanko/wixqa_kb_corpus_chunk_500_overlap_30'), pr_revision=None, pr_num=None)

#### bdanko/wixqa_kb_corpus_chunk_300_overlap_30

In [23]:
fixed_results = []

for article in articles:
    article_id = article.get("id")
    clean_text = clean_wixqa_text(article.get("contents", ""))

    if not clean_text:
        continue

    f_chunks = _fixed_chunks(clean_text, chunk_size=300, overlap=30)
    for i, text in enumerate(f_chunks):
        fixed_results.append(Chunk(
            chunk_id=f"{article_id}_f{i}",
            article_id=article_id,
            text=text,
            metadata={"method": "fixed"}
        ))

print(f"Fixed Chunks: {len(fixed_results)}")

from datasets import Dataset

data_list = [
    {
        "chunk_id": c.chunk_id,
        "article_id": c.article_id,
        "text": c.text,
        "method": c.metadata.get("method")
    }
    for c in fixed_results
]

dataset = Dataset.from_list(data_list)

dataset.push_to_hub("bdanko/wixqa_kb_corpus_chunk_300_overlap_30", private=False)

Fixed Chunks: 55361


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/56 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  47%|####6     | 3.89MB / 8.33MB            

CommitInfo(commit_url='https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_300_overlap_30/commit/e2f32eab1f8cba0a5f5ce1010b9668edd9bbf84d', commit_message='Upload dataset', commit_description='', oid='e2f32eab1f8cba0a5f5ce1010b9668edd9bbf84d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_300_overlap_30', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bdanko/wixqa_kb_corpus_chunk_300_overlap_30'), pr_revision=None, pr_num=None)

#### bdanko/wixqa_kb_corpus_chunk_300_overlap_60

In [24]:
fixed_results = []

for article in articles:
    article_id = article.get("id")
    clean_text = clean_wixqa_text(article.get("contents", ""))

    if not clean_text:
        continue

    f_chunks = _fixed_chunks(clean_text, chunk_size=300, overlap=60)
    for i, text in enumerate(f_chunks):
        fixed_results.append(Chunk(
            chunk_id=f"{article_id}_f{i}",
            article_id=article_id,
            text=text,
            metadata={"method": "fixed"}
        ))

print(f"Fixed Chunks: {len(fixed_results)}")

from datasets import Dataset

data_list = [
    {
        "chunk_id": c.chunk_id,
        "article_id": c.article_id,
        "text": c.text,
        "method": c.metadata.get("method")
    }
    for c in fixed_results
]

dataset = Dataset.from_list(data_list)

dataset.push_to_hub("bdanko/wixqa_kb_corpus_chunk_300_overlap_60", private=False)

Fixed Chunks: 61870


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/62 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  46%|####5     | 3.91MB / 8.58MB            

CommitInfo(commit_url='https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_300_overlap_60/commit/671c79ec29045991f883d41dc6701f03e7e1f030', commit_message='Upload dataset', commit_description='', oid='671c79ec29045991f883d41dc6701f03e7e1f030', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_chunk_300_overlap_60', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bdanko/wixqa_kb_corpus_chunk_300_overlap_60'), pr_revision=None, pr_num=None)

#### Semantic chunking

bdanko/wixqa_kb_corpus_all-MiniLM-L6-v2_semantic

In [14]:
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# we are using all-MiniLM-L6-v2
# because it's lightweight and fast to iterate
# but there are better leaderboard options
model = SentenceTransformer('all-MiniLM-L6-v2')
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [8]:
all_sentences = []
sentence_metadata = [] # To map sentences back to articles

for article in articles:
    content = clean_wixqa_text(article.get("contents", ""))
    if not content: continue

    article_sentences = get_sentences(content)
    for s in article_sentences:
        all_sentences.append(s)
        sentence_metadata.append(article['id'])

In [9]:
len(all_sentences)

166597

In [10]:
all_embeddings = model.encode(
    all_sentences,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/651 [00:00<?, ?it/s]

#### Merging sentence embeddings

We are doing so based on a threshold of 0.5, though this should be experimented with

In [15]:
def merge_semantic_chunks(sentences, embeddings, threshold=0.4, min_size=300, max_size=800):
    chunks = []
    current_text = ""

    distances = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        distances.append(1 - sim)

    for i, sentence in enumerate(sentences):
        current_text = f"{current_text} {sentence}".strip()
        word_count = len(current_text.split())

        is_last = (i == len(sentences) - 1)

        # break check
        if not is_last:
            dist = distances[i]

            # we break if
            # Similarity threshold met, we are above min size
            # oe approaching the hard max size
            if (dist > threshold and word_count >= min_size) or word_count >= max_size:
                chunks.append(current_text)
                current_text = ""
        else:

            # last sentence:
            # if current chunk is too small, append to the previous one
            if word_count < min_size and chunks:
                chunks[-1] = f"{chunks[-1]} {current_text}".strip()
            else:
                chunks.append(current_text)

    return chunks

In [16]:
import numpy as np
from collections import defaultdict

# group indices by article_id
article_to_indices = defaultdict(list)
for idx, a_id in enumerate(sentence_metadata):
    article_to_indices[a_id].append(idx)

semantic_results = []

for article_id, indices in article_to_indices.items():

    # Extract this article's sentences and their corresponding embeddings
    article_sentences = [all_sentences[i] for i in indices]
    article_embeddings = all_embeddings[indices]

    # Apply our merge function
    merged_texts = merge_semantic_chunks(article_sentences, article_embeddings, threshold=0.5)

    for i, text in enumerate(merged_texts):
        semantic_results.append(Chunk(
            chunk_id=f"{article_id}_sem{i}",
            article_id=article_id,
            text=text,
            metadata={"method": "semantic_miniLM"}
        ))

print(f"Total semantic chunks: {len(semantic_results)}")

Total semantic chunks: 8917


In [17]:
# not exactly the distribution we want, but
# it will function, we reduced 166597 sentences to 8917

import pandas as pd
word_counts = [len(c.text.split()) for c in semantic_results]
stats = pd.Series(word_counts)
print("Distribution:")
print(stats.describe())

Distribution:
count    8917.000000
mean      271.599305
std       161.008413
min         4.000000
25%       100.000000
50%       307.000000
75%       357.000000
max      1343.000000
dtype: float64


In [18]:
for chunk in semantic_results[:5]:
  print(f"\nChunk:\n {chunk.text}")


Chunk:
 Wix Events: About the Event Details and Registration Form Pages Guests visiting your site view the events you offer on the Events List page. From there they can learn more on the Events Details page (if enabled) and complete the booking on the Registration Form Page. You can customize how these pages look to suit your events. Tips: Customizations you make to the Events Details page and the Registration Form page (e. g. changing colors and fonts, hiding page elements) affect these page for all events. If your site only has events without tickets, you have the option of disabling the Event Details Page. Events Details Page Site visitors who click to register for an event are first directed to the Event Details page (unless you hid the page). There, guests can receive more complete information about the event. You can customize the page to show or hide various elements, such as a map to the event location and an "About the Event" description. Events without tickets Events without

In [19]:
from datasets import Dataset

semantic_data = [
    {
        "chunk_id": c.chunk_id,
        "article_id": c.article_id,
        "text": c.text,
        "method": c.metadata.get("method")
    }
    for c in semantic_results
]

semantic_dataset = Dataset.from_list(semantic_data)

repo_id = "bdanko/wixqa_kb_corpus_all-MiniLM-L6-v2_semantic"
semantic_dataset.push_to_hub(repo_id, private=False)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   7%|7         |  528kB / 7.21MB            

CommitInfo(commit_url='https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_all-MiniLM-L6-v2_semantic/commit/d0022ba2fce9ce93414802e2eeb79acefbbf3490', commit_message='Upload dataset', commit_description='', oid='d0022ba2fce9ce93414802e2eeb79acefbbf3490', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bdanko/wixqa_kb_corpus_all-MiniLM-L6-v2_semantic', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bdanko/wixqa_kb_corpus_all-MiniLM-L6-v2_semantic'), pr_revision=None, pr_num=None)

### Compute Embeddings and use a vector store

In [4]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

!hf auth login --token {HF_TOKEN}

A new version of huggingface_hub (1.7.1) is available! You are using version 1.6.0.
To update, run: pip install -U huggingface_hub

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `all!` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `all!`


In [5]:
import torch
from sentence_transformers import SentenceTransformer
from datasets import Dataset
import numpy as np
import torch
import random

seed = 15179996
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
# Ensure deterministic behavior for some internal algorithms
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



In [6]:
# we're using all-MiniLM-L6-v2 for embeddings
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
# helper function for once we compute embeddings
# + uploading them to huggingface
def process_and_push(chunk_list, repo_name):
    data = {
        "chunk_id": [c.chunk_id for c in chunk_list],
        "article_id": [c.article_id for c in chunk_list],
        "text": [c.text for c in chunk_list],
        "method": [c.metadata.get("method") for c in chunk_list]
    }

    ds = Dataset.from_dict(data)

    def embed_func(batch):
        batch["embeddings"] = model.encode(batch["text"], convert_to_numpy=True)
        return batch

    ds_with_embeddings = ds.map(embed_func, batched=True, batch_size=128)

    ds_with_embeddings.push_to_hub(f"bdanko/{repo_name}", private=False)
    return ds_with_embeddings

#### Set up the database

In [4]:
!apt-get update
!apt-get install -y postgresql postgresql-server-dev-all build-essential git

# install pgvector
!git clone https://github.com/pgvector/pgvector.git
%cd pgvector
!make
!make install

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,436 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubunt

In [5]:
!service postgresql start

 * Starting PostgreSQL 14 database server
   ...done.


In [6]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"

ALTER ROLE


In [7]:
import psycopg2

conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    host="localhost",
    password="postgres"
)

cur = conn.cursor()
# cur.execute("CREATE EXTENSION vector ;")
conn.commit()

In [8]:
import psycopg2
from pgvector.psycopg2 import register_vector

def setup_db(table_name):
    conn = psycopg2.connect("host=localhost dbname=postgres user=postgres password=postgres")
    cur = conn.cursor()

    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    register_vector(conn)

    # Create table for specific ablation
    cur.execute(f"DROP TABLE IF EXISTS {table_name}")
    cur.execute(f"""
        CREATE TABLE {table_name} (
            id SERIAL PRIMARY KEY,
            chunk_id TEXT,
            article_id TEXT,
            text TEXT,
            embedding VECTOR(384) -- all-MiniLM-L6-v2 dimension
        );
    """)

    # Create HNSW index
    cur.execute(f"CREATE INDEX ON {table_name} USING hnsw (embedding vector_cosine_ops);")

    conn.commit()
    return conn, cur

In [15]:
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# we are using all-MiniLM-L6-v2
# because it's lightweight and fast to iterate
# but there are better leaderboard options
model = SentenceTransformer('all-MiniLM-L6-v2')
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [9]:
from psycopg2.extras import execute_values

def ingest_dataset(ds, table_name, cur, conn):

    # prepare embeddings data
    data_list = [
        (r['chunk_id'], r['article_id'], r['text'], r['embeddings'])
        for r in ds
    ]

    insert_query = f"INSERT INTO {table_name} (chunk_id, article_id, text, embedding) VALUES %s"

    execute_values(cur, insert_query, data_list)
    conn.commit()
    print(f"Ingested {len(data_list)} rows into {table_name}.")

def retrieve_context(query, table_name, k, cur):
    query_vec = model.encode(query)
    # Cosine distance retrieval
    cur.execute(f"""
        SELECT text FROM {table_name}
        ORDER BY embedding <=> %s
        LIMIT %s
    """, (query_vec, k))
    return [row[0] for row in cur.fetchall()]

In [14]:
# initialize DB Table
TABLE_NAME = "wixqa_500_60"
conn, cur = setup_db(TABLE_NAME)

In [15]:
# load chunk size 500, overlap 60
from datasets import load_dataset
ds_500_60 = load_dataset("bdanko/wixqa_kb_corpus_chunk_500_overlap_60", split="train")

README.md:   0%|          | 0.00/388 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.89M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/34954 [00:00<?, ? examples/s]

In [16]:
# add embeddings
def add_embeddings(batch):
    batch["embeddings"] = model.encode(batch["text"])
    return batch

ds_ready = ds_500_60.map(add_embeddings, batched=True, batch_size=128)

Parameter 'function'=<function add_embeddings at 0x7bb8ff873d80> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/34954 [00:00<?, ? examples/s]

In [17]:
# ingest
ingest_dataset(ds_ready, TABLE_NAME, cur, conn)

Ingested 34954 rows into wixqa_500_60.


### Evaluate Chunk Size 500 Overlap 60 K 3

In [19]:
import os
import re
import asyncio
from openai import OpenAI, AsyncOpenAI

# the OpenAI API protocol for API calls
# easier to interface with openrouter
class OpenAIClient:
    def __init__(self, model_name="gpt-4o", base_url=None, api_key=None, extra_headers=None):
        self.api_key = api_key
        self.model_name = model_name
        self.client = OpenAI(
            api_key=self.api_key,
            base_url=base_url,
            default_headers=extra_headers
        )
        self.async_client = AsyncOpenAI(
            api_key=self.api_key,
            base_url=base_url,
            default_headers=extra_headers
        )

    def generate(self, system_message, user_prompt, temperature=0.0, **kwargs):
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=temperature,
                **kwargs
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"OpenAI API error: {e}")
            return None

    async def async_generate(self, system_message, user_prompt, temperature=0.0, **kwargs):
        try:
            response = await self.async_client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=temperature,
                **kwargs
            )
            content = response.choices[0].message.content
            return content.strip() if content else "" # Handle None content safely
        except Exception as e:
            print(f"OpenAI Async API error: {e}")
            return f"ERROR_API_FAILURE" # Return string instead of None

class CerebrasClient(OpenAIClient):
    def __init__(self, model_name="gpt-oss-120b", api_key=None):
        api_key = api_key or os.environ.get("CEREBRAS_API_KEY")
        base_url = "https://api.cerebras.ai/v1"

        super().__init__(
            model_name=model_name,
            base_url=base_url,
            api_key=api_key
        )

In [20]:
import os
from google.colab import userdata

cerebras_api_key = userdata.get('CEREBRAS_KEY')
os.environ['CEREBRAS_KEY'] = cerebras_api_key

client_oss = CerebrasClient(model_name="gpt-oss-120b", api_key=cerebras_api_key)


In [21]:
qa_synthetic = load_qa_split("synthetic")
pd.DataFrame(qa_synthetic).head()

Loading WixQA-synthetic from HuggingFace…


test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/6221 [00:00<?, ? examples/s]

6221 QA pairs loaded.


,question,answer,article_ids
0,How can I manage inventory updates after creat...,When you mark the merchandise as received in t...,[9c2c7c359c723590c8fe2c0e2d49abb9cfdc95990524d...
1,How can I create and send purchase orders usin...,We're excited to announce that it's now possib...,[898b43cf76a908dd88e43830ff4e4244f1bc66f0217f2...
2,How can I manage the activation status of a co...,To activate or deactivate a coupon in Wix Stor...,[77418f552bd01f346fb4724d59fd19ab76a123e4293a7...
3,How do I delete a coupon in Wix Stores?,"To delete a coupon in Wix Stores, follow these...",[ee454a57b80eb0861452a30221236d6dddd57d79dc442...
4,How do I customize the Facebook Like button's ...,To change the layout of your Facebook Like but...,[e4ff804aa8944214be31be99142fd2d1c05cd8eaf2275...


In [22]:
import asyncio
import collections
import openai
from tqdm.asyncio import tqdm_asyncio
from evaluate import load
import json

rouge_metric = load("rouge")

In [23]:
def safe_retrieve(query, table_name, k, connection, model):
    with connection.cursor() as local_cur:
        query_vec = model.encode(query)
        local_cur.execute(f"""
            SELECT text FROM {table_name}
            ORDER BY embedding <=> %s LIMIT %s
        """, (query_vec, k))
        return [row[0] for row in local_cur.fetchall()]


async def run_retrieval_stage(qa_dataset, table_name, k, connection):
    retrieval_sem = asyncio.Semaphore(15)

    async def get_context(idx, item):
        async with retrieval_sem:
            try:
                contexts = await asyncio.to_thread(
                    safe_retrieve, item["question"], table_name, k, connection, model
                )
                return {
                    "original_index": idx,
                    "question": item.get("question"),
                    "answer": item.get("answer"),
                    "article_ids": item.get("article_ids"),
                    "context": "\n".join(contexts)
                }
            except Exception as e:
                return {"original_index": idx, "context": "", "error": str(e)}

    return await tqdm_asyncio.gather(
        *[get_context(i, row) for i, row in enumerate(qa_dataset)],
        desc="Retrieval Stage"
    )

In [35]:
# actual generation of response using API
async def run_generation_stage(retrieved_data, client, output_file, rpm_limit=30):
    sem = asyncio.Semaphore(60)
    results = []

    async def generate_one(item):
        async with sem:
            await asyncio.sleep(60.0 / rpm_limit)
            prompt = f"Context:\n{item['context']}\n\nQuestion: {item['question']}\n\nAnswer:"

            prediction = await client.async_generate(
                system_message="Answer strictly based on context.",
                user_prompt=prompt,
                max_completion_tokens=512
            )

            # Ensure prediction is a string
            item["prediction"] = prediction if prediction else ""

            with open(output_file, "a") as f:
                f.write(json.dumps(item) + "\n")
            return item

    tasks = [generate_one(row) for row in retrieved_data]
    return await tqdm_asyncio.gather(*tasks, desc="Generation Stage")

In [52]:
qa_synthetic = load_qa_split("synthetic")
retrieved_data = await run_retrieval_stage(qa_synthetic, "wixqa_500_60", 3, conn)

Loading WixQA-synthetic from HuggingFace…
6221 QA pairs loaded.


Retrieval Stage: 100%|██████████| 6221/6221 [01:21<00:00, 76.58it/s]


In [53]:
final_results = await run_generation_stage(
    retrieved_data,
    client_oss,
    "/content/eval_final-wixqa_500_60_k3.jsonl"
)

Generation Stage: 100%|██████████| 6221/6221 [06:09<00:00, 16.82it/s]


In [28]:
import numpy as np
from evaluate import load

def compute_lexical_metrics(jsonl_path):
    df = pd.read_json(jsonl_path, lines=True)

    # all api failures are dropped
    df = df[df['prediction'] != "ERROR_API_FAILURE"].dropna(subset=['prediction', 'answer'])

    predictions = df['prediction'].tolist()
    references = df['answer'].tolist()

    # ROUGE Score
    rouge = load("rouge")
    rouge_results = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    # Token-level F1 Score
    #  measures token overlap between
    # the generated answer and the ground-truth
    f1_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens = pred.lower().split()
        ref_tokens = ref.lower().split()

        common = collections.Counter(pred_tokens) & collections.Counter(ref_tokens)
        num_same = sum(common.values())

        if num_same == 0:
            f1_scores.append(0.0)
            continue

        precision = 1.0 * num_same / len(pred_tokens)
        recall = 1.0 * num_same / len(ref_tokens)
        f1 = (2 * precision * recall) / (precision + recall)
        f1_scores.append(f1)

    return {
        "rouge1": rouge_results["rouge1"],
        "avg_f1": np.mean(f1_scores)
    }

In [54]:
metrics = compute_lexical_metrics("/content/eval_final-wixqa_500_60_k3.jsonl")
print(f"Metrics: {metrics}")

Metrics: {'rouge1': np.float64(0.4511669892450101), 'avg_f1': np.float64(0.35742423133799234)}


In [38]:
import json
import asyncio
from tqdm.asyncio import tqdm_asyncio

async def run_context_recall(results_df, client, output_file, rpm_limit=30):
    sem = asyncio.Semaphore(60)

    judge_system_prompt = (
        "You are an objective QA judge. Given a context and a ground-truth answer, "
        "determine if the context contains enough information to derive that answer. "
        "Respond ONLY with 'YES' or 'NO'."
    )

    async def evaluate_recall(row, f_out):
        async with sem:
            await asyncio.sleep(60.0 / rpm_limit)

            user_prompt = (
                f"Question: {row['question']}\n"
                f"Ground Truth Answer: {row['answer']}\n"
                f"Retrieved Context: {row['context']}\n"
                "Verdict (YES/NO):"
            )

            verdict = await client.async_generate(
                system_message=judge_system_prompt,
                user_prompt=user_prompt,
                temperature=0.0
            )

            clean_verdict = verdict.strip().upper()
            score = 1 if "YES" in clean_verdict else 0

            record = {
                "question": row['question'],
                "verdict": clean_verdict,
                "score": score
            }
            f_out.write(json.dumps(record) + "\n")
            return score

    with open(output_file, 'w', encoding='utf-8') as f:
        tasks = [evaluate_recall(row, f) for _, row in results_df.iterrows()]
        verdicts = await tqdm_asyncio.gather(*tasks, desc="Context Recall Judge")

    recall_score = sum(verdicts) / len(verdicts) if verdicts else 0
    return recall_score

In [55]:
results_df = pd.read_json("/content/eval_final-wixqa_500_60_k3.jsonl", lines=True)
recall_score = await run_context_recall(results_df, client_oss, "/content/recall_results-wixqa_500_60_k3.jsonl")
print(f"Context Recall @ K=3: {recall_score:.4f}")

Context Recall Judge:  65%|██████▌   | 4052/6221 [03:09<01:08, 31.60it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Requests per minute limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Requests per minute limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Requests per minute limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Requests per minute limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Requests per minute limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': '

Context Recall Judge: 100%|██████████| 6221/6221 [06:15<00:00, 16.55it/s]

Context Recall @ K=3: 0.6560


### Evaluate Chunk Size 500 Overlap 60 K 1

In [56]:
retrieved_data = await run_retrieval_stage(qa_synthetic, "wixqa_500_60", 1, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:17<00:00, 79.82it/s]


In [57]:
final_results = await run_generation_stage(
    retrieved_data,
    client_oss,
    "/content/eval_final-wixqa_500_60_k1.jsonl"
)

Generation Stage: 100%|██████████| 6221/6221 [06:16<00:00, 16.52it/s]


In [58]:
metrics = compute_lexical_metrics("/content/eval_final-wixqa_500_60_k1.jsonl")
print(f"Metrics: {metrics}")

Metrics: {'rouge1': np.float64(0.4419483360527936), 'avg_f1': np.float64(0.3565842032968538)}


In [59]:
results_df = pd.read_json("/content/eval_final-wixqa_500_60_k1.jsonl", lines=True)
recall_score = await run_context_recall(results_df, client_oss, "/content/recall_results-wixqa_500_60_k1.jsonl")
print(f"Context Recall @ K=1: {recall_score:.4f}")

Context Recall Judge: 100%|██████████| 6221/6221 [05:42<00:00, 18.17it/s]

Context Recall @ K=1: 0.4790


### Evaluate Chunk Size 500 Overlap 60 K 5

In [60]:
retrieved_data = await run_retrieval_stage(qa_synthetic, "wixqa_500_60", 5, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:24<00:00, 73.96it/s]


In [61]:
final_results = await run_generation_stage(
    retrieved_data,
    client_oss,
    "/content/eval_final-wixqa_500_60_k5.jsonl"
)

Generation Stage: 100%|██████████| 6221/6221 [06:13<00:00, 16.64it/s]


In [62]:
metrics = compute_lexical_metrics("/content/eval_final-wixqa_500_60_k5.jsonl")
print(f"Metrics: {metrics}")

Metrics: {'rouge1': np.float64(0.4504653189038761), 'avg_f1': np.float64(0.3529175759231383)}


In [63]:
results_df = pd.read_json("/content/eval_final-wixqa_500_60_k5.jsonl", lines=True)
recall_score = await run_context_recall(results_df, client_oss, "/content/recall_results-wixqa_500_60_k5.jsonl")
print(f"Context Recall @ K=1: {recall_score:.4f}")

Context Recall Judge:  81%|████████  | 5021/6221 [04:35<00:34, 35.07it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  96%|█████████▋| 5994/6221 [05:35<00:07, 28.67it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [06:42<00:00, 15.47it/s]

Context Recall @ K=1: 0.7348


### Evaluate Chunk Size 500 Overlap 60 K 10

In [64]:
retrieved_data = await run_retrieval_stage(qa_synthetic, "wixqa_500_60", 10, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:26<00:00, 71.75it/s]


In [65]:
final_results = await run_generation_stage(
    retrieved_data,
    client_oss,
    "/content/eval_final-wixqa_500_60_k10.jsonl"
)

Generation Stage: 100%|██████████| 6221/6221 [08:19<00:00, 12.45it/s]


In [66]:
metrics = compute_lexical_metrics("/content/eval_final-wixqa_500_60_k10.jsonl")
print(f"Metrics: {metrics}")

Metrics: {'rouge1': np.float64(0.446997251189534), 'avg_f1': np.float64(0.3467236035591771)}


In [67]:
results_df = pd.read_json("/content/eval_final-wixqa_500_60_k10.jsonl", lines=True)
recall_score = await run_context_recall(results_df, client_oss, "/content/recall_results-wixqa_500_60_k10.jsonl")
print(f"Context Recall @ K=10: {recall_score:.4f}")

Context Recall Judge:  37%|███▋      | 2279/6221 [02:40<02:40, 24.54it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  37%|███▋      | 2290/6221 [02:41<02:27, 26.69it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  37%|███▋      | 2299/6221 [02:41<02:14, 29.15it/s]

...

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  97%|█████████▋| 6022/6221 [08:23<00:24,  8.15it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [09:33<00:00, 10.86it/s]

Context Recall @ K=10: 0.8158


### Chunk Size 500 Overlap 30 Database setup

In [68]:
TABLE_NAME_30 = "wixqa_500_30"
conn, cur = setup_db(TABLE_NAME_30)

In [69]:
from datasets import load_dataset
ds_500_30 = load_dataset("bdanko/wixqa_kb_corpus_chunk_500_overlap_30", split="train")

README.md:   0%|          | 0.00/388 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32911 [00:00<?, ? examples/s]

In [70]:
def add_embeddings(batch):
    batch["embeddings"] = model.encode(batch["text"])
    return batch

In [71]:
ds_ready_30 = ds_500_30.map(add_embeddings, batched=True, batch_size=128)

Map:   0%|          | 0/32911 [00:00<?, ? examples/s]

In [72]:
ingest_dataset(ds_ready_30, TABLE_NAME_30, cur, conn)

Ingested 32911 rows into wixqa_500_30.


### Chunk Size 500 Overlap 30 K 1

In [73]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 1, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:25<00:00, 72.97it/s]


In [74]:
output_path_30 = "/content/eval_final-wixqa_500_30_k1.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage: 100%|██████████| 6221/6221 [06:07<00:00, 16.92it/s]


In [75]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (500_30 @ K=1): {metrics_30}")

Lexical Metrics (500_30 @ K=1): {'rouge1': np.float64(0.44119343980090886), 'avg_f1': np.float64(0.35659564405502675)}


In [76]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_500_30_k1.jsonl"
)
print(f"Context Recall @ K=1 (500_30): {recall_score_30:.4f}")

Context Recall Judge: 100%|██████████| 6221/6221 [05:29<00:00, 18.86it/s]

Context Recall @ K=1 (500_30): 0.4773


### Chunk Size 500 Overlap 30 K 3

In [77]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 3, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:20<00:00, 77.49it/s]


In [78]:
output_path_30 = "/content/eval_final-wixqa_500_30_k3.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage: 100%|██████████| 6221/6221 [06:07<00:00, 16.93it/s]


In [79]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (500_30 @ K=3): {metrics_30}")

Lexical Metrics (500_30 @ K=3): {'rouge1': np.float64(0.44981845230869455), 'avg_f1': np.float64(0.3556646452180822)}


In [80]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_500_30_k3.jsonl"
)
print(f"Context Recall @ K=3 (500_30): {recall_score_30:.4f}")

Context Recall Judge: 100%|██████████| 6221/6221 [05:39<00:00, 18.30it/s]

Context Recall @ K=3 (500_30): 0.6512


### Chunk Size 500 Overlap 30 K 5

In [81]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 5, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:27<00:00, 70.82it/s]


In [82]:
output_path_30 = "/content/eval_final-wixqa_500_30_k5.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage:  85%|████████▍ | 5276/6221 [04:23<00:28, 33.09it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage: 100%|██████████| 6221/6221 [05:29<00:00, 18.86it/s]


In [83]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (500_30 @ K=5): {metrics_30}")

Lexical Metrics (500_30 @ K=5): {'rouge1': np.float64(0.45057815475547935), 'avg_f1': np.float64(0.3531358846234095)}


In [84]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_500_30_k5.jsonl"
)
print(f"Context Recall @ K=5 (500_30): {recall_score_30:.4f}")

Context Recall Judge:  50%|█████     | 3115/6221 [02:41<01:34, 33.02it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_e

Context Recall Judge:  50%|█████     | 3127/6221 [02:41<01:21, 38.04it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  66%|██████▌   | 4086/6221 [03:41<01:20, 26.56it/s]

...

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  82%|████████▏ | 5121/6221 [04:43<00:27, 40.68it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [06:43<00:00, 15.43it/s]

Context Recall @ K=5 (500_30): 0.7324


### Chunk Size 500 Overlap 30 K 10

In [85]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 10, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:21<00:00, 76.54it/s]


In [86]:
output_path_30 = "/content/eval_final-wixqa_500_30_k10.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage:  62%|██████▏   | 3862/6221 [04:26<03:43, 10.54it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}
OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  62%|██████▏   | 3877/6221 [04:27<04:24,  8.86it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  62%|██████▏   | 3881/6221 [04:27<04:11,  9.31it/s]

...

Generation Stage:  63%|██████▎   | 3912/6221 [04:30<03:19, 11.57it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  63%|██████▎   | 3922/6221 [04:31<02:15, 16.93it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage: 100%|██████████| 6221/6221 [08:11<00:00, 12.66it/s]


In [87]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (500_30 @ K=10): {metrics_30}")

Lexical Metrics (500_30 @ K=10): {'rouge1': np.float64(0.44557265755173503), 'avg_f1': np.float64(0.346136045779178)}


In [88]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_500_30_k10.jsonl"
)
print(f"Context Recall @ K=10 (500_30): {recall_score_30:.4f}")

Context Recall Judge:  29%|██▉       | 1817/6221 [02:03<01:14, 58.86it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_e

Context Recall Judge:  51%|█████     | 3157/6221 [04:04<01:14, 40.88it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_e

Context Recall Judge:  51%|█████     | 3181/6221 [04:04<01:13, 41.25it/s]

...

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  98%|█████████▊| 6113/6221 [08:24<00:05, 19.75it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [09:27<00:00, 10.97it/s]

Context Recall @ K=10 (500_30): 0.8132


### Chunk Size 300 Overlap 30 Database setup

In [89]:
TABLE_NAME_30 = "wixqa_300_30"
conn, cur = setup_db(TABLE_NAME_30)

In [90]:
from datasets import load_dataset
ds_300_30 = load_dataset("bdanko/wixqa_kb_corpus_chunk_300_overlap_30", split="train")

README.md:   0%|          | 0.00/388 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.33M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/55361 [00:00<?, ? examples/s]

In [91]:
def add_embeddings(batch):
    batch["embeddings"] = model.encode(batch["text"])
    return batch

In [92]:
ds_ready_30 = ds_300_30.map(add_embeddings, batched=True, batch_size=128)

Map:   0%|          | 0/55361 [00:00<?, ? examples/s]

In [93]:
ingest_dataset(ds_ready_30, TABLE_NAME_30, cur, conn)

Ingested 55361 rows into wixqa_300_30.


### Chunk Size 300 Overlap 30 K 1

In [94]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 1, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:19<00:00, 78.45it/s]


In [95]:
output_path_30 = "/content/eval_final-wixqa_300_30_k1.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage: 100%|██████████| 6221/6221 [06:16<00:00, 16.51it/s]


In [96]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (300_30 @ K=1): {metrics_30}")

Lexical Metrics (300_30 @ K=1): {'rouge1': np.float64(0.42498314499283385), 'avg_f1': np.float64(0.3413859767646691)}


In [97]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_300_30_k1.jsonl"
)
print(f"Context Recall @ K=1 (300_30): {recall_score_30:.4f}")

Context Recall Judge: 100%|██████████| 6221/6221 [05:27<00:00, 19.02it/s]

Context Recall @ K=1 (300_30): 0.4064


### Chunk Size 300 Overlap 30 K 3

In [98]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 3, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:30<00:00, 68.70it/s]


In [99]:
output_path_30 = "/content/eval_final-wixqa_300_30_k3.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage: 100%|██████████| 6221/6221 [06:17<00:00, 16.48it/s]


In [100]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (300_30 @ K=3): {metrics_30}")

Lexical Metrics (300_30 @ K=3): {'rouge1': np.float64(0.4366036067780801), 'avg_f1': np.float64(0.34728079148835683)}


In [101]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_300_30_k3.jsonl"
)
print(f"Context Recall @ K=3 (300_30): {recall_score_30:.4f}")

Context Recall Judge: 100%|██████████| 6221/6221 [06:59<00:00, 14.81it/s]

Context Recall @ K=3 (300_30): 0.5613


### Chunk Size 300 Overlap 30 K 5

In [102]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 5, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:20<00:00, 77.42it/s]


In [103]:
output_path_30 = "/content/eval_final-wixqa_300_30_k5.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage: 100%|██████████| 6221/6221 [06:13<00:00, 16.66it/s]


In [104]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (300_30 @ K=5): {metrics_30}")

Lexical Metrics (300_30 @ K=5): {'rouge1': np.float64(0.43763364580840114), 'avg_f1': np.float64(0.3440574396819597)}


In [105]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_300_30_k5.jsonl"
)
print(f"Context Recall @ K=5 (300_30): {recall_score_30:.4f}")

Context Recall Judge:  44%|████▍     | 2762/6221 [02:03<02:29, 23.20it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_e

Context Recall Judge:  69%|██████▊   | 4276/6221 [03:22<02:55, 11.09it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [05:57<00:00, 17.42it/s]

Context Recall @ K=5 (300_30): 0.6521


### Chunk Size 300 Overlap 30 K 10

In [106]:
retrieved_data_30 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_30, 10, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:18<00:00, 78.84it/s]


In [107]:
output_path_30 = "/content/eval_final-wixqa_300_30_k10.jsonl"
final_results_30 = await run_generation_stage(
    retrieved_data_30,
    client_oss,
    output_path_30
)

Generation Stage:  11%|█         | 656/6221 [00:20<03:44, 24.77it/s]

OpenAI Async API error: Error code: 422 - {'message': 'Timed out while updating the structured output state machine', 'type': 'invalid_request_error', 'param': 'response_format', 'code': 'wrong_api_format', 'id': ''}


Generation Stage: 100%|██████████| 6221/6221 [06:25<00:00, 16.13it/s]


In [108]:
metrics_30 = compute_lexical_metrics(output_path_30)
print(f"Lexical Metrics (300_30 @ K=10): {metrics_30}")

Lexical Metrics (300_30 @ K=10): {'rouge1': np.float64(0.43627207332792817), 'avg_f1': np.float64(0.3386978669494063)}


In [109]:
results_df_30 = pd.read_json(output_path_30, lines=True)
recall_score_30 = await run_context_recall(
    results_df_30,
    client_oss,
    "/content/recall_results-wixqa_300_30_k10.jsonl"
)
print(f"Context Recall @ K=10 (300_30): {recall_score_30:.4f}")

Context Recall Judge:  35%|███▌      | 2186/6221 [02:02<02:13, 30.18it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}
OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_e

Context Recall Judge:  82%|████████▏ | 5112/6221 [05:31<00:44, 24.95it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge:  96%|█████████▌| 5964/6221 [06:31<00:09, 26.75it/s]

OpenAI Async API error: Error code: 429 - {'message': 'Tokens per minute limit exceeded - too many tokens processed.', 'type': 'too_many_tokens_error', 'param': 'quota', 'code': 'token_quota_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [07:39<00:00, 13.54it/s]

Context Recall @ K=10 (300_30): 0.7444


### Chunk Size 300 Overlap 60 Database setup

In [10]:
TABLE_NAME_60 = "wixqa_300_60"
conn, cur = setup_db(TABLE_NAME_60)

In [11]:
from datasets import load_dataset
ds_300_60 = load_dataset("bdanko/wixqa_kb_corpus_chunk_300_overlap_60", split="train")

README.md:   0%|          | 0.00/388 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.58M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61870 [00:00<?, ? examples/s]

In [12]:
def add_embeddings(batch):
    batch["embeddings"] = model.encode(batch["text"])
    return batch

In [16]:
ds_ready_60 = ds_300_60.map(add_embeddings, batched=True, batch_size=128)

Parameter 'function'=<function add_embeddings at 0x781d2e33b240> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/61870 [00:00<?, ? examples/s]

In [17]:
ingest_dataset(ds_ready_60, TABLE_NAME_60, cur, conn)

Ingested 61870 rows into wixqa_300_60.


### Chunk Size 300 Overlap 60 K 1

In [25]:
retrieved_data_60 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_60, 1, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:18<00:00, 79.52it/s]


In [26]:
output_path_60 = "/content/eval_final-wixqa_300_60_k1.jsonl"
final_results_60 = await run_generation_stage(
    retrieved_data_60,
    client_oss,
    output_path_60
)

Generation Stage:  19%|█▊        | 1165/6221 [00:43<03:10, 26.54it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  23%|██▎       | 1406/6221 [00:56<02:30, 32.04it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  23%|██▎       | 1431/6221 [00:58<04:09, 19.22it/s]

...

Generation Stage:  94%|█████████▍| 5858/6221 [05:12<00:25, 14.04it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}
OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  95%|█████████▍| 5883/6221 [05:14<00:28, 11.77it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}
OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage: 100%|██████████| 6221/6221 [05:55<00:00, 17.50it/s]


In [30]:
metrics_60 = compute_lexical_metrics(output_path_60)
print(f"Lexical Metrics (300_60 @ K=1): {metrics_60}")

Lexical Metrics (300_60 @ K=1): {'rouge1': np.float64(0.4264717297724376), 'avg_f1': np.float64(0.34304146621889586)}


In [33]:
results_df_60 = pd.read_json(output_path_60, lines=True)
recall_score_60 = await run_context_recall(
    results_df_60,
    client_oss,
    "/content/recall_results-wixqa_300_60_k1.jsonl"
)
print(f"Context Recall @ K=1 (300_60): {recall_score_60:.4f}")

Context Recall Judge:  37%|███▋      | 2292/6221 [02:19<03:35, 18.27it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}
OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge:  37%|███▋      | 2298/6221 [02:19<04:31, 14.44it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge:  37%|███▋      | 2308/6221 [02:20<04:12, 15.51it/s]

...

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge:  39%|███▉      | 2454/6221 [02:31<04:17, 14.61it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [06:05<00:00, 17.04it/s]

Context Recall @ K=1 (300_60): 0.4083


### Chunk Size 300 Overlap 60 K 3

In [34]:
retrieved_data_60 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_60, 3, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:20<00:00, 77.08it/s]


In [36]:
output_path_60 = "/content/eval_final-wixqa_300_60_k3.jsonl"
final_results_60 = await run_generation_stage(
    retrieved_data_60,
    client_oss,
    output_path_60
)

Generation Stage:  44%|████▍     | 2742/6221 [02:32<07:20,  7.90it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  44%|████▍     | 2750/6221 [02:33<06:44,  8.58it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  44%|████▍     | 2760/6221 [02:34<03:18, 17.46it/s]

...

Generation Stage:  55%|█████▌    | 3425/6221 [03:32<01:45, 26.47it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage:  85%|████████▍ | 5257/6221 [05:31<00:33, 28.94it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage: 100%|██████████| 6221/6221 [06:47<00:00, 15.26it/s]


In [37]:
metrics_60 = compute_lexical_metrics(output_path_60)
print(f"Lexical Metrics (300_60 @ K=3): {metrics_60}")

Lexical Metrics (300_60 @ K=3): {'rouge1': np.float64(0.4401619261276508), 'avg_f1': np.float64(0.350297165351954)}


In [39]:
results_df_60 = pd.read_json(output_path_60, lines=True)
recall_score_60 = await run_context_recall(
    results_df_60,
    client_oss,
    "/content/recall_results-wixqa_300_60_k3.jsonl"
)
print(f"Context Recall @ K=3 (300_60): {recall_score_60:.4f}")

Context Recall Judge: 100%|██████████| 6221/6221 [05:39<00:00, 18.31it/s]

Context Recall @ K=3 (300_60): 0.5687


### Chunk Size 300 Overlap 60 K 5

In [40]:
retrieved_data_60 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_60, 5, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:17<00:00, 80.10it/s]


In [41]:
output_path_60 = "/content/eval_final-wixqa_300_60_k5.jsonl"
final_results_60 = await run_generation_stage(
    retrieved_data_60,
    client_oss,
    output_path_60
)

Generation Stage:  86%|████████▌ | 5364/6221 [04:41<01:15, 11.29it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Generation Stage: 100%|██████████| 6221/6221 [05:29<00:00, 18.87it/s]


In [42]:
metrics_60 = compute_lexical_metrics(output_path_60)
print(f"Lexical Metrics (300_60 @ K=5): {metrics_60}")

Lexical Metrics (300_60 @ K=5): {'rouge1': np.float64(0.44046460727914943), 'avg_f1': np.float64(0.3464886954559746)}


In [43]:
results_df_60 = pd.read_json(output_path_60, lines=True)
recall_score_60 = await run_context_recall(
    results_df_60,
    client_oss,
    "/content/recall_results-wixqa_300_60_k5.jsonl"
)
print(f"Context Recall @ K=5 (300_60): {recall_score_60:.4f}")

Context Recall Judge:  18%|█▊        | 1120/6221 [01:16<11:46,  7.22it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge:  19%|█▉        | 1169/6221 [01:20<09:24,  8.96it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge:  19%|█▉        | 1188/6221 [01:21<06:28, 12.95it/s]

...

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge:  36%|███▌      | 2223/6221 [02:27<02:40, 24.92it/s]

OpenAI Async API error: Error code: 503 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}


Context Recall Judge: 100%|██████████| 6221/6221 [05:52<00:00, 17.67it/s]

Context Recall @ K=5 (300_60): 0.6554


### Chunk Size 300 Overlap 60 K 10

In [44]:
retrieved_data_60 = await run_retrieval_stage(qa_synthetic, TABLE_NAME_60, 10, conn)

Retrieval Stage: 100%|██████████| 6221/6221 [01:18<00:00, 79.07it/s]


In [45]:
output_path_60 = "/content/eval_final-wixqa_300_60_k10.jsonl"
final_results_60 = await run_generation_stage(
    retrieved_data_60,
    client_oss,
    output_path_60
)

Generation Stage: 100%|██████████| 6221/6221 [07:07<00:00, 14.54it/s]


In [46]:
metrics_60 = compute_lexical_metrics(output_path_60)
print(f"Lexical Metrics (300_60 @ K=10): {metrics_60}")

Lexical Metrics (300_60 @ K=10): {'rouge1': np.float64(0.4375859594545266), 'avg_f1': np.float64(0.34026081410162573)}


In [47]:
results_df_60 = pd.read_json(output_path_60, lines=True)
recall_score_60 = await run_context_recall(
    results_df_60,
    client_oss,
    "/content/recall_results-wixqa_300_60_k10.jsonl"
)
print(f"Context Recall @ K=10 (300_60): {recall_score_60:.4f}")

Context Recall Judge: 100%|██████████| 6221/6221 [08:08<00:00, 12.73it/s]

Context Recall @ K=10 (300_60): 0.7553
